# Floor Guard AI — Roboflow Spill Classifier Training (Google Colab GPU)
Train a custom PyTorch MobileNetV3 classifier on the **Roboflow Spill Detection Dataset** (`ag-spill/spill-detection-zlhyl`).

- **Target Classes**: `spill` vs `floor floor floor` (normal floor)
- **Task**: Binary image classification
- **Architecture**: MobileNetV3-Small (Transfer Learning)

In [ ]:
# Step 1: Install Dependencies
!pip install -q roboflow torchvision timm torch

In [ ]:
# Step 2: Authenticate and Download Roboflow Dataset
import os
from roboflow import Roboflow

# Set your Roboflow API key here
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("ag-spill").project("spill-detection-zlhyl")
dataset = project.version(1).download("folder", location="roboflow_spill_dataset")
print("Downloaded Roboflow dataset to roboflow_spill_dataset/")

In [ ]:
# Step 3: Train MobileNetV3 Classifier
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Training Device:", device)

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'valid': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

# Load MobileNetV3-Small
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)
print("Model configured. Ready to train for 25 epochs.")

In [ ]:
# Step 4: Export Trained Checkpoint
torch.save(model.state_dict(), "floorguard_spill_classifier_mobilenet.pt")
print("Saved checkpoint to floorguard_spill_classifier_mobilenet.pt")